# OUARTI — Superviseur local autonome

Un seul **REGLAGES**, puis **Tout executer**. Aucun appel a Codex, a une API de modele ou a un service payant. Le superviseur applique des regles fixes : evaluer, entrainer, archiver, comparer, conserver le meilleur, puis passer a la lecon suivante si les criteres sont atteints.

Parcours : 30 m, 100 m, 300 m, 1000 m, puis un tour complet. Un tour exige une progression nette d'au moins une longueur de piste, un compteur de tours augmente et aucun `LapInvalidated` depuis le reset. Un simple passage sur la ligne n'est pas une reussite. Le premier tour est signale, puis des evaluations repetees sont requises avant validation finale.

Les lecons longues utilisent des lots PPO plus longs : garder 128 pas avec reset entre lots empecherait d'apprendre un tour entier. Chaque lecon conserve ses agents et versions. Un candidat en regression est conserve pour inspection, mais le cycle suivant repart du meilleur agent archive.

Les rapports et l'etat persistant sont dans `Code/reports/superviseur_local/<CAMPAGNE>/`. Relancer la meme campagne reprend son etat. Changer le parcours ou les criteres exige une nouvelle campagne. Les budgets de temps et de cycles peuvent etre ajustes au redemarrage. Les notifications sont un fichier local et un signal sonore ; aucun message ne m'est envoye.

Garder le PC allume, le jeu visible, la session locale ouverte et un seul client de conduite. Un probleme technique est retente un nombre limite de fois puis entraine l'arret. Les seuils de capteurs ne sont pas relaches. Le budget peut etre depasse pendant un appel bloque ou une sauvegarde ; il est controle entre les phases et pendant les actions. Pour arreter immediatement : **Interrompre le noyau**. Pas de garantie de maitrise du circuit en une nuit.

Les lecons et leurs objectifs particuliers doivent etre repris avec ce superviseur. Les notebooks manuels precedents restent utiles pour les stages standards 10/30/100 m. Une serie au meme depart mesure sa repetabilite, pas la generalisation.


In [1]:
# REGLAGES — un seul lancement manuel, puis enchainement local.
ACTIVER_SUPERVISEUR = True
ACTIVER_SAUVEGARDE_CHECKPOINT = True  # True si depart absent, voiture deja immobile
CAMPAGNE = 'vers_un_tour_01'
MODELE_SOURCE = 'ppo_vision_checkpoint_v3_stage2'
TRACK_ATTENDU = 'ks_barcelona-layout_gp'
VOITURE_ATTENDUE = 'abarth500'
DIRECTION_PROGRESSION = 1
INTERVALLE_ACTION_S = .12
PENALITE_SORTIE_PISTE = 100.0
DUREE_MAX_HEURES = 8.0
MAX_CYCLES_PAR_LANCEMENT = 12
MAX_CYCLES_PAR_LECON = 8
MAX_CYCLES_SANS_PROGRES = 4
ACTIONS_PAR_CYCLE = 4096  # arrondi au multiple du lot, au moins un lot
EPISODES_EVALUATION = 10
REUSSITE_MIN_PCT = 90.0
SORTIES_MAX_PCT = 5.0
EVALUATIONS_CONSECUTIVES_REQUISES = 2
TOURS_COMPLETS_MINIMUM_PAR_EVALUATION = 3
MAX_REESSAIS_TECHNIQUES = 1
ATTENTE_REESSAI_S = 10.0
GAIN_TEMPS_MIN_PCT = 3.0
DEVICE_EVALUATION = 'cpu'
SIGNAL_SONORE = True
LECONS = [
    {'id':'30m',   'stage':2,'goal_m':30.0,  'episode_seconds':60, 'n_steps':512},
    {'id':'100m',  'stage':3,'goal_m':100.0, 'episode_seconds':120,'n_steps':1024},
    {'id':'300m',  'stage':3,'goal_m':300.0, 'episode_seconds':240,'n_steps':2048},
    {'id':'1000m', 'stage':3,'goal_m':1000.0,'episode_seconds':480,'n_steps':4096},
    {'id':'tour',  'stage':4,'goal_m':None,  'episode_seconds':900,'n_steps':8192},
]


## Definitions locales — aucune conduite au chargement

In [2]:
from pathlib import Path
import sys
CWD=Path.cwd().resolve()
REPO=next((p for p in [CWD,*CWD.parents] if (p/'assetto_corsa_gym'/'assetto_corsa_gym'/'AssettoCorsaPlugin').is_dir()),None)
if REPO is None:
    raise FileNotFoundError('Ouvrir la racine IA_projet_Ouarti_ds_github ou un sous-dossier dans VS Code.')
PROJECT=REPO/'Code'
GYM_ROOT=REPO/'assetto_corsa_gym'/'assetto_corsa_gym'
assert PROJECT.is_dir(), 'Dossier Code introuvable'
if sys.version_info[:2] != (3,9):
    raise RuntimeError('Selectionner le kernel Python 3.9.13 dans VS Code. Actuel : '+sys.version)
for name in ['datasets','models','reports']:
    (PROJECT/name).mkdir(parents=True, exist_ok=True)
print('Depot :',REPO,'\nCode :',PROJECT,'\nPython :',sys.executable)

Depot : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github 
Code : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Code 
Python : c:\Users\elkat\AppData\Local\Programs\Python\Python39\python.exe


In [3]:
import socket, json, ast, time
HOST='127.0.0.1'; PORT_UDP=2345; PORT_TCP=2347

def management(command,timeout=4):
    if command not in ('get_config','get_static_info'):
        raise ValueError('Cette fonction attend une commande qui retourne des donnees.')
    with socket.create_connection((HOST,PORT_TCP),timeout=timeout) as s:
        s.settimeout(timeout)
        s.sendall(command.encode('ascii'))
        data=b''
        deadline=time.monotonic()+timeout
        while time.monotonic()<deadline and len(data)<1024*1024:
            try: chunk=s.recv(65536)
            except socket.timeout: break
            if not chunk: break
            data+=chunk
            try: return json.loads(data)
            except (ValueError,UnicodeError): pass
            try: return ast.literal_eval(data.decode('utf8'))
            except (ValueError,SyntaxError,UnicodeError): pass
    raise RuntimeError('Reponse management absente/incomplete pour '+command+': '+repr(data[:150]))

def request_reset():
    # Le protocole sensors_par NE renvoie AUCUNE reponse a la commande reset.
    with socket.create_connection((HOST,PORT_TCP),timeout=4) as s:
        s.settimeout(4)
        s.sendall(b'reset')
        # Fermer la connexion : le serveur traite la commande sans ACK.

def open_telemetry(timeout=6):
    sock=socket.socket(socket.AF_INET,socket.SOCK_DGRAM)
    sock.settimeout(.30)
    sock.connect((HOST,PORT_UDP))
    end=time.monotonic()+timeout
    try:
        while time.monotonic()<end:
            sock.send(b'connect')
            try: data=sock.recv(2**18)
            except socket.timeout: continue
            if data==b'identified': return sock
        raise TimeoutError('Le plugin UDP 2345 ne repond pas : lancer une session de conduite.')
    except Exception:
        sock.close()
        raise

def next_packet(sock,timeout=3,after_steps=None):
    deadline=time.monotonic()+timeout
    while time.monotonic()<deadline:
        try: data=sock.recv(2**18)
        except socket.timeout: continue
        if data==b'disconnect': raise ConnectionError('Connexion telemetrie interrompue')
        if data==b'identified': continue
        try: packet=json.loads(data)
        except (ValueError,UnicodeError): continue
        if not isinstance(packet,dict): continue
        if after_steps is not None and int(packet.get('steps',-1))<=int(after_steps): continue
        packet['_received_unix']=time.time()
        return packet
    raise TimeoutError('Aucun nouvel echantillon telemetrie dans le delai imparti.')

def close_telemetry(sock):
    if sock is None: return
    try: sock.send(b'disconnect')
    except OSError: pass
    sock.close()

In [4]:
import sys, time, numpy as np, cv2
PLUGIN_PATH=GYM_ROOT/'AssettoCorsaPlugin'/'plugins'/'sensors_par'
if not PLUGIN_PATH.is_dir():
    raise FileNotFoundError('Plugin Python source manquant : '+str(PLUGIN_PATH))
if str(PLUGIN_PATH) not in sys.path:
    sys.path.insert(0,str(PLUGIN_PATH))
from dual_buffer import GrabberSharedMemoryDualBuffer

class CameraReader:
    def __init__(self,config,age_limit=.55):
        if not config.get('screen_capture_enable'):
            raise RuntimeError('screen_capture_enable=False dans le plugin installe.')
        self.cam=GrabberSharedMemoryDualBuffer(int(config['final_image_width']),
                      int(config['final_image_height']),str(config['color_mode']))
        self.age_limit=float(age_limit)
        self.last_seq=-1
    def read(self,timeout=2,require_new=True):
        deadline=time.monotonic()+float(timeout)
        last_err='Aucune image stable et fraiche'
        while time.monotonic()<deadline:
            try:
                seq,stamp,arr=self.cam.get_timed_image()
                age=time.time()-float(stamp)
                if age<-.15 or age>self.age_limit:
                    last_err='Image perimee ou horloge incoherente (age %.3f s)'%age
                elif require_new and int(seq)<=self.last_seq:
                    last_err='Image non actualisee (seq %s)'%seq
                else:
                    self.last_seq=int(seq)
                    return int(seq),float(stamp),arr
            except (RuntimeError,ValueError) as e:
                last_err=str(e)
            time.sleep(.015)
        raise TimeoutError('Capture indisponible : '+last_err)

def frame_to_image84(frame):
    if frame.ndim==3:
        # La capture couleur du plugin est en BGR.
        gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    elif frame.ndim==2:
        gray=frame
    else:
        raise ValueError('Format camera invalide : '+str(frame.shape))
    gray=cv2.resize(gray,(84,84),interpolation=cv2.INTER_AREA)
    return np.ascontiguousarray(gray[np.newaxis,:,:],dtype=np.uint8)

In [5]:
import numpy as np, pandas as pd
from scipy.spatial import cKDTree

class TrackGeometry:
    def __init__(self,path):
        df=pd.read_csv(path)
        required=['left_border_x','left_border_y','right_border_x','right_border_y']
        if not set(required).issubset(df.columns):
            raise ValueError('CSV geometrie non compatible, colonnes attendues: '+str(required))
        left=df[['left_border_x','left_border_y']].to_numpy(dtype=np.float64)
        right=df[['right_border_x','right_border_y']].to_numpy(dtype=np.float64)
        self.center=(left+right)*.5
        self.width=np.linalg.norm(left-right,axis=1)
        self.delta=np.roll(self.center,-1,axis=0)-self.center
        self.segment=np.linalg.norm(self.delta,axis=1)
        if (self.segment>.001).sum()<10:
            raise ValueError('CSV de circuit invalide: pas assez de points')
        self.arc=np.concatenate(([0.],np.cumsum(self.segment)))
        self.length=float(self.arc[-1])
        self.tree=cKDTree(self.center)
    def project(self,raw):
        xy=np.asarray([float(raw['world_position_x']),float(raw['world_position_y'])],dtype=np.float64)
        _,j=self.tree.query(xy)
        j=int(j)
        forward=self.delta[j]/max(float(self.segment[j]),1e-8)
        signed=float(forward[0]*(xy[1]-self.center[j,1])-forward[1]*(xy[0]-self.center[j,0]))
        heading=float(np.arctan2(forward[1],forward[0]))
        return float(self.arc[j]),signed,float(self.width[j]),heading

def numerical_state(raw,track):
    arc,offset,width,heading=track.project(raw)
    diff=(float(raw.get('yaw',0))-heading+np.pi)%(2*np.pi)-np.pi
    vec=[float(raw.get('speed',0))/100.,float(raw.get('accelX',0))/3.,
         float(raw.get('accelY',0))/3.,float(raw.get('angular_velocity_y',0))/4.,
         float(np.clip(offset/max(width/2.,1.),-3,3)),np.sin(diff),np.cos(diff),
         float(raw.get('accStatus',0)),float(raw.get('brakeStatus',0))]
    return np.clip(np.asarray(vec,dtype=np.float32),-5,5),arc,offset,width

def find_track_file(track_name):
    # Les fichiers .pkl sont des grilles d'occupation; ici la recompense
    # utilise les CSV bord gauche/bord droit du depot assetto_corsa_gym.
    name=str(track_name)
    path=GYM_ROOT/'AssettoCorsaConfigs'/'tracks'/(name+'.csv')
    if not path.is_file():
        raise FileNotFoundError('CSV de geometrie introuvable : '+str(path))
    return path

In [6]:
import ctypes, threading, time, numpy as np
from pathlib import Path

class VJoySafe:
    """Pilote local Windows deja valide dans 04_Test_Volant_vJoy/05_Test_Pedales."""
    X=0x30; Y=0x31; Z=0x32
    def __init__(self,device_id=1,watchdog_sec=.7):
        dll_path=Path(r'C:\Program Files\vJoy\x64\vJoyInterface.dll')
        if not dll_path.exists(): raise FileNotFoundError(str(dll_path))
        self.dll=ctypes.CDLL(str(dll_path)); self.device_id=int(device_id)
        self.dll.GetVJDStatus.argtypes=[ctypes.c_uint]
        self.dll.GetVJDStatus.restype=ctypes.c_int
        self.dll.AcquireVJD.argtypes=[ctypes.c_uint]
        self.dll.AcquireVJD.restype=ctypes.c_int
        self.dll.SetAxis.argtypes=[ctypes.c_long,ctypes.c_uint,ctypes.c_uint]
        self.dll.SetAxis.restype=ctypes.c_int
        self.dll.RelinquishVJD.argtypes=[ctypes.c_uint]
        self.dll.RelinquishVJD.restype=None
        status=self.dll.GetVJDStatus(self.device_id)
        if status!=1: raise RuntimeError('vJoy peripherique %d indisponible, statut=%d'%(self.device_id,status))
        if not self.dll.AcquireVJD(self.device_id): raise RuntimeError('Echec AcquireVJD')
        self.lock=threading.Lock(); self.active=True
        self.last_action=time.monotonic(); self.watchdog_sec=float(watchdog_sec)
        self.last_command={'steer_cmd':0.0, 'throttle_cmd':0.0, 'brake_cmd':1.0}
        self._set_axes(16384,0,32767)  # frein de securite lors de la prise de controle
        self.thread=threading.Thread(target=self._watchdog,daemon=True)
        self.thread.start()
    def _set_axes(self,x,y,z):
        with self.lock:
            for axis,value in ((self.X,x),(self.Y,y),(self.Z,z)):
                if not self.dll.SetAxis(int(value),self.device_id,axis):
                    raise RuntimeError('Echec SetAxis vJoy sur '+hex(axis))
    def action(self,a):
        a=np.asarray(a,dtype=np.float32).reshape(-1)
        if a.shape!=(3,) or not np.isfinite(a).all(): raise ValueError('Action PPO invalide')
        steer=float(np.clip(a[0],-1.,1.))
        gas=float(np.clip(a[1],0.,1.))
        brake=float(np.clip(a[2],0.,1.))
        # Politique PPO au depart aleatoire : accelerateur et frein peuvent etre positifs.
        # Une faible demande de frein ne doit plus annuler un fort accelerateur.
        # Commandes longitudinales mutuellement exclusives : retenir la plus forte.
        if gas >= brake:
            brake = 0.0
        else:
            gas = 0.0
        self._set_axes(round((steer+1)*16383.5),round(gas*32767),round(brake*32767))
        # Ces valeurs sont les commandes *effectivement envoyees* a vJoy.
        self.last_command = {'steer_cmd':steer, 'throttle_cmd':gas, 'brake_cmd':brake}
        self.last_action=time.monotonic()
    def hold(self):
        if self.active:
            self._set_axes(16384,0,32767)
            self.last_action=time.monotonic()
    def _watchdog(self):
        while self.active:
            if time.monotonic()-self.last_action>self.watchdog_sec:
                try: self.hold()
                except Exception: pass
            time.sleep(.06)
    def close(self):
        if not self.active: return
        try:self.hold()
        finally:
            self.active=False
            self.thread.join(timeout=.3)
            self.dll.RelinquishVJD(self.device_id)

In [7]:
# Cellule de pont entre Jupyter et l'application CSP Lua OUARTICheckpoint.
import json, os, time, uuid, math
from pathlib import Path

BRIDGE_DIR = Path.home() / 'Documents' / 'Assetto Corsa' / 'OUARTI_checkpoint_v3'
ANCHOR_FILE = BRIDGE_DIR / 'anchor.json'

class CheckpointBridge:
    def __init__(self, directory=BRIDGE_DIR):
        self.directory = Path(directory)
        self.command = self.directory / 'command.txt'
        self.response = self.directory / 'status.txt'

    def request(self, command, timeout=8.0):
        if command not in ('PING', 'STATUS', 'SAVE', 'LOAD', 'CAPABILITIES'):
            raise ValueError('Commande CSP inconnue: ' + command)
        if not self.directory.is_dir():
            raise RuntimeError('Application CSP OUARTICheckpoint absente : ' + str(self.directory))
        token = uuid.uuid4().hex
        temporary = self.directory / ('command_' + token + '.tmp')
        temporary.write_text(token + '|' + command, encoding='ascii')
        os.replace(str(temporary), str(self.command))
        deadline = time.monotonic() + float(timeout)
        last = ''
        while time.monotonic() < deadline:
            try:
                last = self.response.read_text(encoding='utf8').strip()
            except (FileNotFoundError, UnicodeError, PermissionError):
                time.sleep(0.05)
                continue
            parts = last.split('|', 2)
            if len(parts) == 3 and parts[0] == token:
                if parts[1] == 'ERROR':
                    raise RuntimeError('CSP checkpoint: ' + parts[2])
                if parts[1] == 'PENDING':
                    time.sleep(0.05)
                    continue
                return parts[1], parts[2]
            time.sleep(0.05)
        raise TimeoutError('Pas de reponse CSP pour ' + command + '. Dernier statut: ' + last[:140])

    def capabilities(self):
        code, detail = self.request('CAPABILITIES')
        if code != 'CAPABILITIES':
            raise RuntimeError('Reponse CAPABILITIES inattendue: ' + code + ' ' + detail)
        fields = {}
        for field in detail.split(';'):
            key, separator, value = field.strip().partition('=')
            if separator:
                fields[key.strip()] = value.strip()
        for key in ('offline', 'full_snapshot', 'position_fallback'):
            if fields.get(key) not in ('true', 'false'):
                raise RuntimeError('Capacite CSP absente ou invalide: ' + key + ': ' + detail)
        return {key: value == 'true' for key, value in fields.items()
                if key in ('offline', 'full_snapshot', 'position_fallback')}

    def assert_supported(self):
        caps = self.capabilities()
        if not caps['offline']:
            raise RuntimeError('Checkpoint reserve a une session locale hors ligne.')
        if not (caps['full_snapshot'] or caps['position_fallback']):
            raise RuntimeError(
                'Checkpoint indisponible: CSP ne fournit ni snapshot complet ni '
                'repositionnement autorise dans cette session. La connexion au pont fonctionne. '
                'Verifier la version CSP dans Content Manager > Settings > Custom Shaders Patch '
                '> About & Updates. Sur le PC examine le 16/11/2026, CSP 0.1.79 est installe '
                'et son SDK ne declare pas saveCarStateAsync/loadCarState. '
                'Apres mise a jour compatible avec sensors_par, relancer le jeu et refaire '
                'PING puis CAPABILITIES. Si les deux capacites restent false, ne pas lancer PPO. '
                'Le fallback position depend aussi de physics.allowed(); '
                'une mise a jour seule ne garantit pas son activation.')
        return caps

    def assert_saved(self):
        code, detail = self.request('STATUS')
        if code != 'SAVED':
            raise RuntimeError('Checkpoint non enregistre dans cette session CSP: ' + detail
                               + '. Executer la cellule de sauvegarde avec ACTIVER_SAUVEGARDE_CHECKPOINT=True, '
                               'voiture immobile, puis remettre False. Refaire SAVE apres chaque redemarrage du jeu.')
        return detail

    def load(self):
        code, detail = self.request('LOAD', timeout=8)
        if code != 'LOAD_REQUESTED':
            raise RuntimeError('CSP n a pas confirme la commande LOAD: ' + code + ' ' + detail)


def telemetry_xy(raw):
    return float(raw['world_position_x']), float(raw['world_position_y'])


def distance_from_anchor(raw, anchor):
    x, y = telemetry_xy(raw)
    xa, ya = float(anchor['x']), float(anchor['y'])
    distance = math.hypot(x - xa, y - ya)
    angle_error = abs((float(raw['yaw']) - float(anchor['yaw']) + math.pi) % (2 * math.pi) - math.pi)
    return distance, angle_error


def make_anchor_from_raw(raw, track, car):
    x, y = telemetry_xy(raw)
    if float(raw.get('speed', 999)) > 0.3:
        raise RuntimeError('Immobiliser la voiture avant sauvegarde (<1 km/h).')
    return {'track': track, 'car': car, 'x': x, 'y': y,
            'yaw': float(raw['yaw']), 'created_at': time.time(),
            'note': 'Checkpoint CSP en memoire, non reutilisable apres redemarrage du jeu.'}


In [8]:
import os, json, time, numpy as np, gymnasium as gym
from gymnasium import spaces

class LiveAssettoVisionEnv(gym.Env):
    """1 instance = 1 jeu local, 1 connexion UDP, 1 vJoy. Jamais de simulation sans jeu."""
    metadata={'render_modes':[]}
    def __init__(self,expected_track,expected_car,episode_seconds=45,
                 action_interval=.12,progress_sign=1,log_file=None,curriculum_stage=1,offtrack_penalty=100.0):
        super().__init__()
        self.offtrack_penalty=float(offtrack_penalty)
        if not np.isfinite(self.offtrack_penalty) or self.offtrack_penalty <= 0:
            raise ValueError('La penalite de sortie doit etre un nombre positif fini.')
        self.expected_track=str(expected_track)
        self.expected_car=str(expected_car)
        self.curriculum_stage=int(curriculum_stage)
        self.goals={1:10.0,2:30.0,3:100.0,4:None}
        if self.curriculum_stage not in self.goals:
            raise ValueError('Etape du programme: 1, 2, 3 ou 4.')
        self.goal_m=OBJECTIF_DISTANCE_M
        self._start_lap=0
        self._lap_crossed=False
        self._lap_invalid=False
        self.episode_progress=0.0
        self.rollout_remaining=None
        self.progress_sign=int(progress_sign)
        if self.progress_sign not in (-1,1): raise ValueError('progress_sign doit valoir -1 ou 1')
        self.episode_seconds=float(episode_seconds)
        self.action_interval=float(action_interval)
        self.action_space=spaces.Box(low=np.array([-1,0,0],dtype=np.float32),
                                     high=np.ones(3,dtype=np.float32),dtype=np.float32)
        self.observation_space=spaces.Dict({
            'image':spaces.Box(low=0,high=255,shape=(1,84,84),dtype=np.uint8),
            'state':spaces.Box(low=-5.,high=5.,shape=(9,),dtype=np.float32)})
        config=management('get_config'); sess=management('get_static_info')
        track=sess.get('TrackName','')
        if sess.get('TrackConfiguration'): track+='-'+str(sess['TrackConfiguration'])
        car=str(sess.get('CarName',''))
        if track!=self.expected_track or car!=self.expected_car:
            raise RuntimeError('Mauvaise session : '+track+'/'+car+' attendu '+self.expected_track+'/'+self.expected_car)
        if bool(config.get('vjoy_executed_by_server')):
            raise RuntimeError('Ce mode utilise vJoy local: laisser vjoy_executed_by_server=False.')
        if not config.get('screen_capture_enable'):
            raise RuntimeError('Capture camera desactivee dans le plugin')
        if sess.get('autoShifterOn') is False:
            raise RuntimeError('Activer la boite automatique dans Assetto Corsa avant PPO.')
        self.track=TrackGeometry(find_track_file(track))
        self.camera=CameraReader(config)
        self.sock=None; self.controller=None
        self.raw=None; self.arc=None;self.damage=0.;self.lap=0
        self.t0=0.;self.last_obs=None;self.idle_since=0.;self.episode_index=0;self.steps=0
        self.last_action=np.zeros(3,np.float32)
        self.log_file=Path(log_file) if log_file else None
        self.bridge=CheckpointBridge()
        self.bridge.request('PING', timeout=5)
        self.bridge.assert_supported()
        self.bridge.assert_saved()
        if not ANCHOR_FILE.is_file():
            raise RuntimeError('Point de depart inconnu : utiliser la cellule SAUVEGARDER CHECKPOINT avant PPO.')
        self.anchor=json.loads(ANCHOR_FILE.read_text(encoding='utf8'))
        if self.anchor.get('track')!=track or self.anchor.get('car')!=car:
            raise RuntimeError('Checkpoint sauvegarde pour une autre voiture ou un autre circuit.')
        if self.log_file: self.log_file.parent.mkdir(parents=True,exist_ok=True)
        print('Config controle local: vJoy=local, plugin',config.get('sampling_freq'),'Hz, camera',config['color_mode'])
    def begin_rollout(self, steps):
        self.rollout_remaining = int(steps)
        if self.rollout_remaining <= 0:
            raise ValueError('Nombre de pas du lot invalide')

    def safe_stop(self):
        if self.controller is not None: self.controller.hold()
    def _connect(self):
        if self.controller is None:
            self.controller=VJoySafe(device_id=1)
        if self.sock is None:
            self.sock=open_telemetry()
    def _drain_pending_telemetry(self, max_packets=4096, max_seconds=0.12):
        """Vide les vieux datagrammes UDP sans bloquer ni perdre le timeout initial.

        A appeler apres le reset et avant la lecture de chaque observation :
        pendant time.sleep(), le buffer UDP conserve des mesures deja perimees.
        """
        if self.sock is None:
            return 0
        timeout_before = self.sock.gettimeout()
        removed = 0
        deadline = time.monotonic() + float(max_seconds)
        try:
            self.sock.setblocking(False)
            while removed < max_packets and time.monotonic() < deadline:
                try:
                    self.sock.recv(2**18)
                    removed += 1
                except (BlockingIOError, InterruptedError):
                    break
        finally:
            self.sock.settimeout(timeout_before)
        return removed

    def _observe(self, after_steps=None):
        """Obtient une image et une telemetrie recentes, avec delai controle.

        Si elles ne correspondent pas, on reprend une PAIRE fraiche plutot
        que de conserver la telemetrie d'un essai precedent.
        """
        deadline = time.monotonic() + 4.0
        last_error = 'Aucune paire image/telemetrie valide'
        while time.monotonic() < deadline:
            self._drain_pending_telemetry()
            remaining = deadline - time.monotonic()
            if remaining <= 0:
                break
            try:
                raw = next_packet(
                    self.sock,
                    timeout=min(0.8, remaining),
                    after_steps=after_steps,
                )
                remaining = deadline - time.monotonic()
                if remaining <= 0:
                    break
                seq, stamp, frame = self.camera.read(
                    timeout=min(0.8, remaining),
                )
            except (TimeoutError, RuntimeError, OSError, ConnectionError) as exc:
                last_error = str(exc)
                continue

            packet_stamp = float(raw.get('timestamp_unix', raw['_received_unix']))
            gap = abs(float(stamp) - packet_stamp)
            if gap > 0.40:
                now = time.time()
                last_error = (
                    'Camera/telemetrie non synchronisees: ecart %.3f s '
                    '(age image %.3f s, age telemetrie %.3f s)'
                    % (gap, now - float(stamp), now - packet_stamp)
                )
                continue

            state, arc, lateral, width = numerical_state(raw, self.track)
            obs = {
                'image': frame_to_image84(frame),
                'state': state,
            }
            return raw, obs, arc, lateral, width, seq, gap
        raise TimeoutError('Impossible de synchroniser les capteurs: ' + last_error)
    def reset(self,*,seed=None,options=None):
        super().reset(seed=seed)
        self._connect()
        self.safe_stop()
        self.bridge.assert_saved()
        # Ici on RESTAURE un snapshot CSP, pas un reset local sans retour au depart.
        self.bridge.load()
        time.sleep(0.45)
        self._drain_pending_telemetry()
        last_err='Pas de telemetrie valide apres restauration'
        deadline=time.monotonic()+10.0
        while time.monotonic()<deadline:
            try:
                raw,o,arc,lat,width,seq,gap=self._observe()
                distance,angle=distance_from_anchor(raw,self.anchor)
                speed=float(raw.get('speed',999.))
                if distance>1.5 or angle>0.30 or speed>0.75:
                    last_err=('Checkpoint non restaure: position %.2f m, orientation %.2f rad, '
                              'vitesse %.2f m/s' % (distance,angle,speed))
                    time.sleep(0.08)
                    continue
                break
            except (TimeoutError,RuntimeError,OSError,ConnectionError,ValueError) as e:
                last_err=str(e)
        else:
            self.safe_stop()
            raise RuntimeError('RESET FIXE REFUSE : '+last_err+'. Entrainer = False.')
        self.raw=raw;self.last_obs=o;self.arc=arc
        self.damage=float(raw.get('damage_total',0))
        self.lap=int(raw.get('LapCount',0))
        self._start_lap=self.lap
        self._lap_crossed=False
        self._lap_invalid=False
        if self.curriculum_stage==4:
            if 'LapCount' not in raw or 'LapInvalidated' not in raw:
                raise RuntimeError('Tour complet : LapCount et LapInvalidated requis.')
            if bool(raw['LapInvalidated']):
                raise RuntimeError('Le checkpoint contient deja un tour invalide. Recreer un depart valide dans une nouvelle session.')
        self.idle_since=time.monotonic()
        self.t0=time.monotonic();self.steps=0;self.episode_index+=1
        self.last_action=np.zeros(3,np.float32)
        self.episode_progress=0.0
        print('Reset fixe confirme: %.2f m / %.2f rad / %.2f m/s' % (distance,angle,speed))
        return o,{'camera_gap_s':gap,'camera_sequence':seq,'track':self.expected_track,
                  'reset_verified':True,'reset_distance_m':distance,'reset_yaw_error_rad':angle,
                  'curriculum_stage':self.curriculum_stage,'goal_distance_m':self.goal_m}
    def step(self,action):
        if self.sock is None or self.raw is None:
            raise RuntimeError('reset() doit etre appele avant step()')
        if self.rollout_remaining is not None:
            self.rollout_remaining -= 1
        rollout_cut = self.rollout_remaining == 0
        action=np.asarray(action,dtype=np.float32)
        try:
            self.controller.action(action)
            time.sleep(self.action_interval)
            raw,o,arc,lateral,width,seq,gap=self._observe(after_steps=self.raw.get('steps'))
            dt=float(raw.get('timestamp_unix',raw['_received_unix']))-float(self.raw.get('timestamp_unix',self.raw['_received_unix']))
            if not .005<dt<1.0: raise RuntimeError('Delta temps telemetrie invalide: %s'%dt)
            move=(arc-self.arc+self.track.length/2)%self.track.length-self.track.length/2
            move*=self.progress_sign
            if abs(move)>max(4.,100.*dt*1.7):
                # Collision, teleportation, mauvaise projection ou horodatage : episode invalide.
                raise RuntimeError('Saut de position non physique (%.1f m sur %.3f s)'%(move,dt))
            tyres=int(raw.get('numberOfTyresOut',0))
            offtrack=bool(tyres>2 or abs(lateral)>width*.5)
            new_damage=float(raw.get('damage_total',0))
            collision=bool(new_damage>self.damage+.02)
            new_lap=int(raw.get('LapCount',0))
            if self.curriculum_stage==4:
                if 'LapCount' not in raw or 'LapInvalidated' not in raw:
                    raise RuntimeError('Compteurs de tour absents.')
                if new_lap<self.lap or new_lap>self.lap+1:
                    raise RuntimeError('Compteur de tours incoherent.')
                self._lap_crossed=self._lap_crossed or new_lap>self.lap
                self._lap_invalid=self._lap_invalid or bool(raw['LapInvalidated'])
            lap_finished=False
            now=time.monotonic(); speed=float(raw.get('speed',0))
            if speed>1.:self.idle_since=now
            idle=bool(now-self.idle_since>6.)
            wrong_way=bool(move<-.25)
            # Curriculum sans imitation : progres geometrie, puis distances croissantes.
            self.episode_progress+=float(move)
            lap_finished=bool(self.curriculum_stage==4 and self._lap_crossed
                and self.episode_progress>=self.track.length and not self._lap_invalid
                and not offtrack and not collision)
            invalid_lap=bool(self.curriculum_stage==4 and self._lap_invalid)
            milestone=bool(self.goal_m is not None and self.episode_progress>=self.goal_m
                           and not offtrack and not collision)
            if self.curriculum_stage==1:
                reward=float(2.5*np.clip(move,-3.,5.)-.02
                             -.006*abs(float(action[0])-float(self.last_action[0]))
                             -self.offtrack_penalty*offtrack-25.*collision-5.*idle-2.*wrong_way
                             +20.*milestone)
            else:
                reward=float(np.clip(move,-5.,8.)-.02
                             -.015*abs(float(o['state'][4]))
                             -.008*abs(float(action[0])-float(self.last_action[0]))
                             -self.offtrack_penalty*offtrack-35.*collision-8.*idle-2.*wrong_way
                             +30.*milestone+70.*(lap_finished and not offtrack and not collision))
            reward-=self.offtrack_penalty*invalid_lap
            terminated=bool(offtrack or collision or idle or milestone or lap_finished or invalid_lap)
            time_limit = bool(now-self.t0>=self.episode_seconds)
            truncated=bool(time_limit or (rollout_cut and not terminated))
            self.last_obs=o;self.raw=raw;self.arc=arc;self.damage=new_damage;self.lap=new_lap
            self.last_action=action.copy();self.steps+=1
            # Enregistrer les commandes reelles et les retours des pedales.
            # Utile pour distinguer exploration PPO et probleme de configuration du jeu.
            sent=dict(self.controller.last_command)
            def telemetry_number(*keys):
                for key in keys:
                    if key in raw and raw[key] is not None:
                        try: return float(raw[key])
                        except (ValueError,TypeError): pass
                return None
            info={'progress_m':float(move),'speed_ms':speed,'offtrack':offtrack,'collision':collision,
                  'offtrack_penalty':float(self.offtrack_penalty*offtrack),
                  'world_position_x':float(raw['world_position_x']),
                  'world_position_y':float(raw['world_position_y']),
                  'yaw':float(raw['yaw']),'lateral_m':float(lateral),'track_width_m':float(width),
                  'lap_finished':lap_finished,'lap_invalidated':invalid_lap,
                  'circuit_length_m':float(self.track.length),'lap_count':new_lap,
                  'lap_count_start':self._start_lap,'idle':idle,'wrong_way':wrong_way,'camera_gap_s':gap,
                  'camera_sequence':seq,'episode_id':self.episode_index,
                  'episode_progress_m':float(self.episode_progress),
                  'milestone_reached':bool(milestone),'goal_distance_m':self.goal_m,
                  'curriculum_stage':self.curriculum_stage,
                  'rollout_cut':bool(rollout_cut and not terminated),
                  'time_limit':time_limit,
                  **sent,
                  'accStatus':telemetry_number('accStatus'),
                  'brakeStatus':telemetry_number('brakeStatus'),
                  'rpm':telemetry_number('rpm','RPM','engineRPM'),
                  'gear':telemetry_number('gear','Gear','currentGear','CurrentGear')}
            if terminated or truncated: self.safe_stop()
            if self.log_file:
                with self.log_file.open('a',encoding='utf8') as f:
                    f.write(json.dumps({'time':time.time(),'reward':reward,'terminated':terminated,
                             'truncated':truncated,**info},ensure_ascii=False)+'\n')
            return o,reward,terminated,truncated,info
        except (OSError,RuntimeError,TimeoutError,ConnectionError,ValueError) as exc:
            self.safe_stop()
            raise RuntimeError('Incident capteur ou observation : '+str(exc)) from exc
    def close(self):
        # Even if the brake command fails, release UDP, vJoy and shared memory.
        errors=[]
        try:self.safe_stop()
        except Exception as exc:errors.append(str(exc))
        if self.sock is not None:
            try:close_telemetry(self.sock)
            except Exception as exc:errors.append(str(exc))
            finally:self.sock=None
        if self.controller is not None:
            try:self.controller.close()
            except Exception as exc:errors.append(str(exc))
            finally:self.controller=None
        for name in ('dual_shm','meta_shm'):
            mapping=getattr(getattr(self.camera,'cam',None),name,None)
            if mapping is not None:
                try:mapping.close()
                except Exception as exc:errors.append(str(exc))
        if errors:raise RuntimeError('Nettoyage evaluation : '+'; '.join(errors))


In [9]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor
import json, time, torch, shutil, uuid

CONTROL_SCHEMA='vjoy_local_xyz_pedales_exclusives_v3'
OBS_SCHEMA='dict_image_1x84x84_state_9'

import hashlib, re


def empreinte_fichier(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):
            digest.update(block)
    return digest.hexdigest()


def prefix_modele(selection):
    path=Path(selection)
    if not path.is_absolute():path=PROJECT/'models'/path
    if path.suffix=='.zip':path=path.with_suffix('')
    return path.resolve()


def fichiers_modele(prefix):
    prefix=Path(prefix)
    return prefix.with_suffix('.zip'),prefix.with_name(prefix.name+'_meta.json')


def lire_modele(selection):
    prefix=prefix_modele(selection);model,metadata=fichiers_modele(prefix)
    if not model.is_file() or not metadata.is_file():
        raise FileNotFoundError('ZIP et metadonnees requis : '+str(prefix))
    versions=(PROJECT/'models'/'versions').resolve()
    if versions in prefix.parents:
        manifest_path=prefix.parent/'manifest.json'
        if not manifest_path.is_file():raise RuntimeError('Archive sans manifeste : '+str(prefix))
        manifest=json.loads(manifest_path.read_text(encoding='utf8'))
        if empreinte_fichier(model)!=manifest.get('zip_sha256') or empreinte_fichier(metadata)!=manifest.get('meta_sha256'):
            raise RuntimeError('Archive alteree, lecture refusee : '+str(prefix))
    return prefix,json.loads(metadata.read_text(encoding='utf8'))


def verifier_destination_modifiable(prefix):
    prefix=Path(prefix).resolve()
    versions=(PROJECT/'models'/'versions').resolve()
    if prefix==versions or versions in prefix.parents:
        raise RuntimeError('Les archives versions sont protegees. Creer un agent de travail sous un nouveau nom.')


def destination_agent(nom):
    if not isinstance(nom,str) or re.fullmatch(r'[A-Za-z0-9_-]+',nom) is None:
        raise ValueError('Nom de modele : lettres, chiffres, tirets et underscores uniquement.')
    prefix=(PROJECT/'models'/nom).resolve()
    verifier_destination_modifiable(prefix)
    return prefix


def archiver_agent(selection,motif='avant_modification'):
    prefix,meta=lire_modele(selection)
    model,metadata=fichiers_modele(prefix)
    hashes={'zip_sha256':empreinte_fichier(model),'meta_sha256':empreinte_fichier(metadata)}
    versions=PROJECT/'models'/'versions'
    # The exact same ZIP/metadata pair only needs one archive.
    for existing in versions.glob('*/*/manifest.json'):
        manifest=json.loads(existing.read_text(encoding='utf8'))
        if all(manifest.get(k)==v for k,v in hashes.items()):
            archived=existing.parent/'agent'
            a,b=fichiers_modele(archived)
            if not a.is_file() or not b.is_file() or empreinte_fichier(a)!=hashes['zip_sha256'] or empreinte_fichier(b)!=hashes['meta_sha256']:
                raise RuntimeError('Archive alteree : '+str(existing.parent))
            return archived
    label=re.sub(r'[^A-Za-z0-9_-]','_',str(meta.get('model',prefix.name)))
    folder=versions/label/(time.strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8])
    folder.mkdir(parents=True,exist_ok=False)
    archived=folder/'agent';a,b=fichiers_modele(archived)
    shutil.copy2(model,a);shutil.copy2(metadata,b)
    if empreinte_fichier(a)!=hashes['zip_sha256'] or empreinte_fichier(b)!=hashes['meta_sha256']:
        raise RuntimeError('Copie archive non conforme : '+str(folder))
    manifest={'version_format':1,'source':str(prefix),'motif':motif,'created_at':time.time(),
        'model':meta.get('model',prefix.name),'curriculum_stage':meta.get('curriculum_stage'),
        'goal_m':meta.get('goal_m'),'total_timesteps':meta.get('total_timesteps'),**hashes}
    (folder/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf8')
    return archived


def creer_agent_depuis(source,nom_cible,stage_cible=None,objectif_lecon='par_defaut'):
    """Fork a verified archived model. Never overwrite an existing target."""
    original,meta=lire_modele(source)
    target=destination_agent(nom_cible)
    out_zip,out_meta=fichiers_modele(target)
    if out_zip.exists() or out_meta.exists():
        raise FileExistsError('Destination deja presente, choisir un nouveau nom : '+str(target))
    if meta.get('control')!='vjoy_local_xyz_pedales_exclusives_v3' or meta.get('observation')!='dict_image_1x84x84_state_9' or meta.get('reset')!='csp_snapshot_verified':
        raise RuntimeError('Source incompatible avec le pilote V3.')
    current=meta.get('curriculum_stage')
    if type(current) is not int or current not in (1,2,3,4):
        raise RuntimeError('Etape source inconnue.')
    stage=current if stage_cible is None else stage_cible
    if type(stage) is not int or stage not in (current,current+1) or stage not in (1,2,3,4):
        raise ValueError('Autorise : copie de la meme etape ou transfert vers la suivante uniquement.')
    archived=archiver_agent(original,'source_transfert' if stage!=current else 'source_restauration')
    source_zip,source_meta=fichiers_modele(archived)
    # Build a new metadata tree; original metadata stays byte-for-byte intact.
    updated={**meta,'model':nom_cible,'curriculum_stage':stage,
        'goal_m':({1:10.,2:30.,3:100.,4:None}[stage] if objectif_lecon=='par_defaut' else objectif_lecon),
        'parent_version':str(archived.relative_to((PROJECT/'models').resolve())),
        'parent_stage':current,'parent_total_timesteps':int(meta['total_timesteps']),
        'initialization':'stage_transfer' if stage!=current else 'restored_copy',
        'created_at':time.time(),'saved_at':time.time()}
    for key in ('run_id','log_dir'):updated.pop(key,None)
    if stage!=current:
        updated['stage_start_timesteps']=int(meta['total_timesteps'])
        updated['stage_timesteps']=0
    # Exclusive creation also prevents accidental replacement on repeated execution.
    out_zip.parent.mkdir(parents=True,exist_ok=True)
    with source_zip.open('rb') as src,out_zip.open('xb') as dst:
        shutil.copyfileobj(src,dst)
    with out_meta.open('x',encoding='utf8') as dst:
        json.dump(updated,dst,indent=2,ensure_ascii=False)
    if empreinte_fichier(out_zip)!=empreinte_fichier(source_zip):
        raise RuntimeError('Le nouvel agent ne correspond pas aux poids sources.')
    # Also freeze the initial state of the child, before its first action.
    initial=archiver_agent(target,'initialisation_agent')
    print('Agent cree :',out_zip,'| stage',stage,'| compteur herite',updated['total_timesteps'])
    print('Source conservee :',archived)
    print('Version initiale conservee :',initial)
    return target


def inventorier_versions():
    rows=[]
    for path in sorted((PROJECT/'models'/'versions').glob('*/*/manifest.json')):
        meta=json.loads(path.read_text(encoding='utf8'))
        rows.append({'modele':meta.get('model'),'stage':meta.get('curriculum_stage'),
            'compteur':meta.get('total_timesteps'),'motif':meta.get('motif'),
            'source_selectionnable':str((path.parent/'agent').relative_to(PROJECT/'models'))})
    return rows



def validate_resume_metadata(meta):
    expected = {'track':TRACK_ATTENDU, 'car':VOITURE_ATTENDUE,
                'control':CONTROL_SCHEMA, 'observation':OBS_SCHEMA,
                'reset':'csp_snapshot_verified',
                'progress_sign':DIRECTION_PROGRESSION,
                'curriculum_stage':ETAPE_APPRENTISSAGE,'goal_m':OBJECTIF_DISTANCE_M}
    mismatch = {key:(meta.get(key), value) for key,value in expected.items()
                if meta.get(key) != value}
    if mismatch:
        raise RuntimeError('Modele incompatible avec cette session/configuration: ' + str(mismatch))


def save_training(agent, prefix, meta):
    # Remplacement atomique de chaque fichier : eviter les ZIP partiellement ecrits.
    prefix=Path(prefix)
    verifier_destination_modifiable(prefix)
    previous_zip,previous_meta=fichiers_modele(prefix)
    if previous_zip.exists() or previous_meta.exists():
        archiver_agent(prefix,'avant_remplacement')
    prefix.parent.mkdir(parents=True,exist_ok=True)
    temporary=prefix.with_name(prefix.name+'_writing_'+uuid.uuid4().hex)
    updated={**meta,'total_timesteps':int(agent.num_timesteps),'saved_at':time.time()}
    if 'stage_start_timesteps' in updated:
        updated['stage_timesteps']=int(agent.num_timesteps)-int(updated['stage_start_timesteps'])
    temporary_meta=temporary.with_name(temporary.name+'_meta.json')
    try:
        agent.save(str(temporary))
        temporary_meta.write_text(json.dumps(updated,indent=2),encoding='utf8')
        os.replace(str(temporary.with_suffix('.zip')),str(prefix.with_suffix('.zip')))
        os.replace(str(temporary_meta),str(prefix.with_name(prefix.name+'_meta.json')))
    finally:
        for leftover in (temporary.with_suffix('.zip'),temporary_meta):
            if leftover.exists(): leftover.unlink()
    if prefix.parent.resolve()==(PROJECT/'models').resolve():
        archiver_agent(prefix,'sauvegarde_agent')
    return updated


class PauseDuringUpdates(BaseCallback):
    """Coupure explicite du lot, frein pendant le calcul, puis reset frais."""
    def __init__(self,core,prefix,meta,save_every=1024):
        super().__init__()
        self.core=core; self.prefix=prefix; self.meta=meta
        self.save_every=int(save_every)
        self.paused=False; self.last_saved=0

    def _on_training_start(self):
        if self.training_env.num_envs != 1:
            raise RuntimeError('Une seule instance Assetto Corsa est supportee.')
        self.last_saved=int(self.model.num_timesteps)

    def _on_rollout_start(self):
        # Sauvegarder avant le reset frais : le temps disque reste hors episode.
        if self.model.num_timesteps-self.last_saved >= self.save_every:
            dest=self.prefix.parent/'checkpoints'/(self.prefix.name+'_'+str(self.model.num_timesteps)+'_steps_'+self.meta['run_id'])
            save_training(self.model,dest,self.meta)
            self.last_saved=int(self.model.num_timesteps)
        if self.paused:
            # VecEnv a deja reset apres la coupure. Refaire le reset APRES le
            # calcul, car le jeu a continue sous frein pendant la mise a jour.
            self.model._last_obs=self.training_env.reset()
            self.model._last_episode_starts=np.ones((1,),dtype=bool)
            self.paused=False
        self.core.begin_rollout(self.model.n_steps)

    def _on_step(self):
        if time.monotonic()>=SUP_DEADLINE:
            self.core.safe_stop()
            return False
        return True

    def _on_rollout_end(self):
        self.core.safe_stop()
        self.paused=True

    def _on_training_end(self):
        self.core.safe_stop()




c:\Users\elkat\AppData\Local\Programs\Python\Python39\lib\site-packages\gym\envs\registration.py:307: DeprecationWarning: The package name gym_minigrid has been deprecated in favor of minigrid. Please uninstall gym_minigrid and install minigrid with `pip install minigrid`. Future releases will be maintained under the new package name minigrid.
  fn()
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [10]:
def sauvegarder_checkpoint_si_absent(bridge, track, car):
    """Creation autorisee par le reglage; ne remplace pas un depart deja valide."""
    code, detail=bridge.request('STATUS')
    if code == 'SAVED' and ANCHOR_FILE.is_file():
        anchor=json.loads(ANCHOR_FILE.read_text(encoding='utf8'))
        if (anchor.get('track'),anchor.get('car')) == (track,car):
            print('Checkpoint deja enregistre : conservation du depart existant.')
            return
        raise RuntimeError('Le fichier anchor.json correspond a une autre session. '
                           'Recreer volontairement le checkpoint dans une nouvelle session du jeu.')
    sock=open_telemetry()
    try:
        sample=next_packet(sock,timeout=3)
        make_anchor_from_raw(sample,track,car)  # exige une voiture immobile
        print('Sauvegarde du checkpoint CSP...')
        code, detail=bridge.request('SAVE',timeout=15)
        if code != 'SAVED':
            raise RuntimeError('Sauvegarde CSP non confirmee : '+code+' '+detail)
        sample=next_packet(sock,timeout=3)
        anchor=make_anchor_from_raw(sample,track,car)
        ANCHOR_FILE.write_text(json.dumps(anchor,indent=2),encoding='utf8')
        bridge.assert_saved()
        print('Checkpoint enregistre :',anchor)
    finally:
        close_telemetry(sock)


In [11]:
from stable_baselines3 import PPO
from IPython.display import display
import hashlib, uuid, traceback


def file_sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):
            digest.update(block)
    return digest.hexdigest()


def verifier_configuration():
    if type(ACTIVER_EVALUATION) is not bool or type(ACTIVER_SAUVEGARDE_CHECKPOINT) is not bool:
        raise ValueError('Les interrupteurs doivent etre True ou False.')
    if not isinstance(MODELES_A_EVALUER,(list,tuple)) or not MODELES_A_EVALUER:
        raise ValueError('Choisir au moins un modele dans MODELES_A_EVALUER.')
    if type(N_EPISODES) is not int or N_EPISODES<=0:
        raise ValueError('N_EPISODES doit etre un entier positif.')
    if type(MAX_ACTIONS_PAR_EPISODE) is not int or MAX_ACTIONS_PAR_EPISODE<=0:
        raise ValueError('MAX_ACTIONS_PAR_EPISODE doit etre un entier positif.')
    if ETAPE_APPRENTISSAGE not in (1,2,3,4) or DIRECTION_PROGRESSION not in (-1,1):
        raise ValueError('Etape ou sens de progression invalide.')
    if not all(np.isfinite(v) and v>0 for v in (DUREE_EPISODE_S,INTERVALLE_ACTION_S,PENALITE_SORTIE_PISTE)):
        raise ValueError('Durees et penalite doivent etre positives et finies.')
    descriptors=[];seen=set()
    expected={'track':TRACK_ATTENDU,'car':VOITURE_ATTENDUE,
              'control':'vjoy_local_xyz_pedales_exclusives_v3',
              'observation':'dict_image_1x84x84_state_9',
              'reset':'csp_snapshot_verified','curriculum_stage':ETAPE_APPRENTISSAGE,
              'progress_sign':DIRECTION_PROGRESSION,'goal_m':OBJECTIF_DISTANCE_M}
    for selected in MODELES_A_EVALUER:
        if not isinstance(selected,str) or not selected.strip():
            raise ValueError('Chaque modele doit etre un chemin non vide.')
        path=Path(selected)
        if not path.is_absolute():path=PROJECT/'models'/path
        if path.suffix!='.zip':path=path.with_suffix('.zip')
        path=path.resolve()
        if path in seen:raise ValueError('Modele selectionne deux fois : '+str(path))
        seen.add(path)
        meta_path=path.with_name(path.stem+'_meta.json')
        if not path.is_file() or not meta_path.is_file():
            raise FileNotFoundError('ZIP et metadonnees requis : '+str(path))
        versions=(PROJECT/'models'/'versions').resolve()
        if versions in path.parents:
            manifest_path=path.parent/'manifest.json'
            if not manifest_path.is_file():raise RuntimeError('Archive sans manifeste : '+str(path))
            manifest=json.loads(manifest_path.read_text(encoding='utf8'))
            if file_sha256(path)!=manifest.get('zip_sha256') or file_sha256(meta_path)!=manifest.get('meta_sha256'):
                raise RuntimeError('Archive alteree : '+str(path))
        meta=json.loads(meta_path.read_text(encoding='utf8'))
        mismatch={k:(meta.get(k),v) for k,v in expected.items() if meta.get(k)!=v}
        if mismatch:raise RuntimeError('Modele incompatible '+path.name+' : '+str(mismatch))
        descriptors.append({'label':str(selected),'path':path,'meta_path':meta_path,
                            'meta':meta,'sha256':file_sha256(path),
                            'meta_sha256':file_sha256(meta_path)})
    return descriptors


def construire_resume(rows):
    if not rows:return pd.DataFrame()
    data=pd.DataFrame(rows);summaries=[]
    for model,group in data.groupby('modele',sort=False):
        valid=group[group['valide']]
        successes=valid[valid['succes']]
        def rate(key):return float(valid[key].mean()*100) if len(valid) else None
        summaries.append({'modele':model,'episodes':len(group),'episodes_valides':len(valid),
            'erreurs_techniques':int((~group['valide']).sum()),
            'reussite_pct':rate('succes'),'sortie_pct':rate('offtrack'),
            'collision_detectee_pct':rate('collision'),'immobilisation_pct':rate('idle'),
            'limite_pct':rate('limite'),
            'tours_complets':int(valid['tour_complet_verifie'].sum()),
            'tours_invalides':int(valid['tour_invalide'].sum()),
            'temps_median_reussite_s':float(successes.duree_s.median()) if len(successes) else None,
            'progression_moyenne_m':float(valid.progression_m.mean()) if len(valid) else None,
            'reward_moyenne':float(valid.reward.mean()) if len(valid) else None,
            'volant_sature_pct':float(valid.volant_sature_pas.sum()/valid.pas.sum()*100) if valid.pas.sum() else None,
            'variation_volant_moyenne':float(valid.variation_volant_totale.sum()/valid.variations_comptees.sum()) if valid.variations_comptees.sum() else None,
            'vitesse_max_kmh':float(valid.vitesse_max_kmh.max()) if len(valid) else None})
    return pd.DataFrame(summaries)


def exporter_evaluation(directory,rows,state):
    directory=Path(directory)
    pd.DataFrame(rows).to_csv(directory/'episodes.csv',index=False)
    summary=construire_resume(rows)
    summary.to_csv(directory/'comparaison.csv',index=False)
    (directory/'etat_session.json').write_text(json.dumps(state,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf8')
    return summary


In [12]:
def entrainer_ppo():
    core=None; agent=None
    prefix=destination_agent(MODELE_NOM)
    meta_file=prefix.with_name(prefix.name+'_meta.json')
    model_file=prefix.with_suffix('.zip')
    run_id=time.strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]
    log_dir=PROJECT/'reports'/('ppo_checkpoint_v3_stage'+str(ETAPE_APPRENTISSAGE))/run_id
    log_dir.mkdir(parents=True,exist_ok=True)
    if not model_file.exists() and MODELE_SOURCE_TRANSFERT and not EXIGER_MODELE_EXISTANT:
        source_prefix,source_meta=lire_modele(MODELE_SOURCE_TRANSFERT)
        if ETAPE_APPRENTISSAGE!=source_meta.get('curriculum_stage',0)+1:
            raise ValueError('Le transfert doit aller vers le stage suivant. Pour revenir, utiliser le notebook de versions.')
        expected_source={**source_meta,'curriculum_stage':ETAPE_APPRENTISSAGE,'goal_m':OBJECTIF_DISTANCE_M}
        validate_resume_metadata(expected_source)
        creer_agent_depuis(source_prefix,MODELE_NOM,ETAPE_APPRENTISSAGE)
    resume=bool(REPRENDRE_MODELE_EXISTANT and model_file.is_file())
    if EXIGER_MODELE_EXISTANT and not resume:
        raise FileNotFoundError('Modele requis pour reprise: '+str(model_file))
    if model_file.exists() and not REPRENDRE_MODELE_EXISTANT:
        raise FileExistsError('Choisir un nouveau MODELE_NOM pour repartir de zero : '+str(model_file))
    previous_meta={}
    if resume:
        previous_meta=json.loads(meta_file.read_text(encoding='utf8'))
        validate_resume_metadata(previous_meta)
        version=archiver_agent(prefix,'avant_reprise')
        print('Version de retour conservee :',version)
    meta={'model':MODELE_NOM,'track':TRACK_ATTENDU,'car':VOITURE_ATTENDUE,
          'progress_sign':DIRECTION_PROGRESSION,'control':CONTROL_SCHEMA,
          'observation':OBS_SCHEMA,'image_color':'gray_from_plugin',
          'curriculum_stage':ETAPE_APPRENTISSAGE,'reset':'csp_snapshot_verified',
          'goal_m':OBJECTIF_DISTANCE_M,
          'episode_boundary':'rollout_truncated_then_reset_after_update',
          'run_id':run_id,'log_dir':str(log_dir),
          'episode_seconds':DUREE_EPISODE_S,'action_interval':INTERVALLE_ACTION_S,
          'reward_version':'superviseur_tour_verifie_v1', 'offtrack_penalty':PENALITE_SORTIE_PISTE}
    for key in ('parent_version','parent_stage','parent_total_timesteps','initialization','created_at','stage_start_timesteps','stage_timesteps'):
        if key in previous_meta:meta[key]=previous_meta[key]
    try:
        core=LiveAssettoVisionEnv(TRACK_ATTENDU,VOITURE_ATTENDUE,
             episode_seconds=DUREE_EPISODE_S,action_interval=INTERVALLE_ACTION_S,
             progress_sign=DIRECTION_PROGRESSION,log_file=log_dir/'ppo_steps.jsonl',
             curriculum_stage=ETAPE_APPRENTISSAGE,offtrack_penalty=PENALITE_SORTIE_PISTE)
        env=Monitor(core,filename=str(log_dir/'monitor.csv'),allow_early_resets=True)
        if resume:
            agent=PPO.load(str(model_file),env=env,device='auto',
                           tensorboard_log=str(log_dir/'tensorboard'),n_steps=PAS_PAR_LOT,batch_size=64)
            print('REPRISE du modele, compteur initial:',agent.num_timesteps)
        else:
            agent=PPO('MultiInputPolicy',env,n_steps=PAS_PAR_LOT,batch_size=64,n_epochs=4,
                      learning_rate=3e-4,gamma=.99,gae_lambda=.95,ent_coef=.005,
                      verbose=1,device='auto',tensorboard_log=str(log_dir/'tensorboard'))
            print('NOUVEAU modele, compteur initial:',agent.num_timesteps)
        warm={key:np.zeros(space.shape,dtype=space.dtype)
              for key,space in agent.observation_space.spaces.items()}
        for _ in range(3):agent.predict(warm,deterministic=True)
        start_steps=int(agent.num_timesteps)
        cb=PauseDuringUpdates(core,prefix,meta)
        agent.learn(total_timesteps=int(TOTAL_TIMESTEPS),reset_num_timesteps=not resume,
                    callback=cb,tb_log_name=MODELE_NOM,progress_bar=False)
        core.safe_stop()
        meta=save_training(agent,prefix,meta)
        print('Modele:',model_file,'metadata:',meta_file)
        print('Actions ajoutees:',agent.num_timesteps-start_steps,'; compteur cumule:',agent.num_timesteps)
        print('Journaux de cette session:',log_dir)
        print('Version finale conservee :',archiver_agent(prefix,'fin_session'))
    except (KeyboardInterrupt,Exception) as exc:
        if core is not None:
            try:core.safe_stop()
            except Exception as stop_error:print('Erreur frein:',stop_error)
        if agent is not None:
            interrupted=PROJECT/'models'/'checkpoints'/(MODELE_NOM+'_interrupted_'+run_id)
            save_training(agent,interrupted,meta)
            print('Arret: sauvegarde separee:',interrupted.with_suffix('.zip'))
        raise
    finally:
        if core is not None: core.close()


In [13]:
def executer_evaluation():
    descriptors=verifier_configuration()
    print('Modeles verifies :')
    for desc in descriptors:
        print(' -',desc['label'],'| compteur apprentissage :',desc['meta'].get('total_timesteps'))
    if not (ACTIVER_EVALUATION or ACTIVER_SAUVEGARDE_CHECKPOINT):
        print('Evaluation desactivee : aucune connexion au jeu, aucune commande.')
        print('Regler ACTIVER_EVALUATION=True puis Tout executer pour mesurer le pilotage.')
        return None
    # Charger tous les modeles avant toute commande vehicule. Aucun learn/save.
    agents=[]
    if ACTIVER_EVALUATION:
        for desc in descriptors:
            agent=PPO.load(str(desc['path']),device=DEVICE_EVALUATION)
            agent.policy.set_training_mode(False)
            if int(agent.num_timesteps)!=int(desc['meta'].get('total_timesteps',-1)):
                raise RuntimeError('Compteur ZIP/metadonnees incoherent : '+desc['label'])
            # Warm up inference before any reset/control: no vehicle action here.
            warm={key:np.zeros(space.shape,dtype=space.dtype)
                  for key,space in agent.observation_space.spaces.items()}
            for _ in range(3):agent.predict(warm,deterministic=True)
            agents.append(agent)
    bridge=CheckpointBridge()
    print('Application :',bridge.request('PING'))
    print('Capacites :',bridge.assert_supported())
    session=management('get_static_info')
    track=session.get('TrackName','')
    if session.get('TrackConfiguration'):track+='-'+str(session['TrackConfiguration'])
    car=str(session.get('CarName',''))
    if (track,car)!=(TRACK_ATTENDU,VOITURE_ATTENDUE):
        raise RuntimeError('Session inattendue : '+track+'/'+car)
    if ACTIVER_SAUVEGARDE_CHECKPOINT:
        sauvegarder_checkpoint_si_absent(bridge,track,car)
    code,detail=bridge.request('STATUS')
    if code!='SAVED' or not ANCHOR_FILE.is_file():
        raise RuntimeError('Checkpoint absent. Immobiliser la voiture et activer '
                           'ACTIVER_SAUVEGARDE_CHECKPOINT dans REGLAGES puis Tout executer.')
    anchor=json.loads(ANCHOR_FILE.read_text(encoding='utf8'))
    if (anchor.get('track'),anchor.get('car'))!=(track,car):
        raise RuntimeError('Reference de checkpoint incompatible avec la session.')
    if not ACTIVER_EVALUATION:
        print('Checkpoint pret. Evaluation desactivee, aucune conduite lancee.')
        return None
    run_id=time.strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]
    directory=PROJECT/'reports'/'evaluation_v3'/run_id
    directory.mkdir(parents=True,exist_ok=False)
    state={'session':run_id,'statut':'en_cours','debut_unix':time.time(),
           'deterministic':True,'apprentissage':False,'coupures_ppo':False,
           'episodes_demandes_par_modele':N_EPISODES,'track':track,'car':car,
           'curriculum_stage':ETAPE_APPRENTISSAGE,'episode_seconds':DUREE_EPISODE_S,
           'goal_m':OBJECTIF_DISTANCE_M,'action_interval':INTERVALLE_ACTION_S,'offtrack_penalty':PENALITE_SORTIE_PISTE,
           'anchor':anchor,'modeles':[{'nom':d['label'],'path':str(d['path']),
               'sha256':d['sha256'],'meta_sha256':d['meta_sha256'],
               'compteur':d['meta']['total_timesteps']} for d in descriptors],
           'limites':['Meme depart fixe, repetitions deterministes : pas une mesure de generalisation.',
                      'Progression geometrique discretisee comme pendant entrainement.',
                      'Aucune collision detectee ne valide pas le detecteur.']}
    rows=[];core=None;active_row=None
    try:
        for model_index,(desc,agent) in enumerate(zip(descriptors,agents),1):
            print('Modele',model_index,'/',len(descriptors),':',desc['label'])
            core=LiveAssettoVisionEnv(track,car,episode_seconds=DUREE_EPISODE_S,
                 action_interval=INTERVALLE_ACTION_S,progress_sign=DIRECTION_PROGRESSION,
                 curriculum_stage=ETAPE_APPRENTISSAGE,offtrack_penalty=PENALITE_SORTIE_PISTE,
                 log_file=directory/('modele_%02d_steps.jsonl'%model_index))
            if agent.observation_space!=core.observation_space or agent.action_space!=core.action_space:
                raise RuntimeError('Espaces incompatibles : '+desc['label'])
            # No begin_rollout() : evaluation episodes have no PPO batch cuts.
            assert core.rollout_remaining is None
            for episode in range(1,N_EPISODES+1):
                if time.monotonic()>=SUP_DEADLINE:raise BudgetAtteint('Budget horaire atteint.')
                active_row={'modele':desc['label'],'episode':episode,'valide':False,'succes':False,
                    'tour_complet_verifie':False,'tour_invalide':False,
                    'motif':'reset_en_cours','erreur':'','pas':0,'reward':0.,'progression_m':0.,
                    'duree_s':0.,'distance_parcourue_m':0.,'vitesse_max_kmh':0.,
                    'offtrack':False,'collision':False,'idle':False,'limite':False,
                    'volant_sature_pas':0,'variation_volant_totale':0.,'variations_comptees':0,
                    'ecart_lateral_max_normalise':0.,'camera_gap_max_s':0.}
                observation,reset_info=core.reset()
                if not reset_info.get('reset_verified'):
                    raise RuntimeError('Reset non verifie : evaluation refusee.')
                active_row.update(reset_distance_m=reset_info['reset_distance_m'],
                                  reset_yaw_error_rad=reset_info['reset_yaw_error_rad'])
                previous_xy=np.array(telemetry_xy(core.raw));previous_steer=None
                started=time.monotonic();done=False
                for _ in range(MAX_ACTIONS_PAR_EPISODE):
                    if time.monotonic()>=SUP_DEADLINE:raise BudgetAtteint('Budget horaire atteint.')
                    action,_=agent.predict(observation,deterministic=True)
                    observation,reward,terminated,truncated,info=core.step(action)
                    if info.get('sensor_error'):
                        raise RuntimeError('Capteurs indisponibles : '+str(info['sensor_error']))
                    if info.get('rollout_cut'):
                        raise RuntimeError('Coupure de lot inattendue pendant evaluation.')
                    active_row['pas']+=1;active_row['reward']+=float(reward)
                    active_row['progression_m']+=float(info.get('progress_m',0.))
                    active_row['vitesse_max_kmh']=max(active_row['vitesse_max_kmh'],float(info.get('speed_ms',0.))*3.6)
                    active_row['camera_gap_max_s']=max(active_row['camera_gap_max_s'],float(info.get('camera_gap_s',0.)))
                    active_row['ecart_lateral_max_normalise']=max(active_row['ecart_lateral_max_normalise'],abs(float(observation['state'][4])))
                    steer=float(info['steer_cmd'])
                    active_row['volant_sature_pas']+=int(abs(steer)>.95)
                    if previous_steer is not None:
                        active_row['variation_volant_totale']+=abs(steer-previous_steer)
                        active_row['variations_comptees']+=1
                    previous_steer=steer
                    xy=np.array(telemetry_xy(core.raw))
                    active_row['distance_parcourue_m']+=float(np.linalg.norm(xy-previous_xy));previous_xy=xy
                    active_row['tour_complet_verifie']=bool(info.get('lap_finished',False))
                    active_row['tour_invalide']=bool(active_row['tour_invalide'] or info.get('lap_invalidated',False))
                    for key in ('offtrack','collision','idle'):
                        active_row[key]=bool(active_row[key] or info.get(key,False))
                    active_row['succes']=bool(info.get('milestone_reached',False) or
                        (ETAPE_APPRENTISSAGE==4 and info.get('lap_finished',False)))
                    active_row['succes']=bool(active_row['succes'] and not active_row['offtrack'] and not active_row['collision'] and not active_row['tour_invalide'])
                    if terminated or truncated:
                        done=True
                        active_row['limite']=bool(truncated)
                        break
                core.safe_stop()
                active_row['duree_s']=time.monotonic()-started
                active_row['valide']=True
                if not done:active_row['limite']=True
                active_row['motif']=('sortie' if active_row['offtrack'] else
                    'collision_detectee' if active_row['collision'] else
                    'succes' if active_row['succes'] else 'immobilisation' if active_row['idle'] else 'limite')
                rows.append(active_row);active_row=None
                exporter_evaluation(directory,rows,state)
                hook=globals().get('SUR_TOUR_CONFIRME')
                if rows[-1]['tour_complet_verifie'] and hook is not None:
                    hook(directory)
                print('Episode',episode,'/',N_EPISODES,':',rows[-1]['motif'],
                      '|',rows[-1]['pas'],'actions |',round(rows[-1]['duree_s'],2),'s')
            core.close();core=None
        state['statut']='terminee'
    except BaseException as exc:
        state['statut']='budget_atteint' if isinstance(exc,BudgetAtteint) else 'interrompue' if isinstance(exc,KeyboardInterrupt) else 'erreur_technique'
        state['erreur']=str(exc) or type(exc).__name__
        if active_row is not None:
            active_row['motif']='interruption' if isinstance(exc,KeyboardInterrupt) else 'erreur_technique'
            active_row['erreur']=state['erreur'];rows.append(active_row)
        (directory/'erreur.txt').write_text(traceback.format_exc(),encoding='utf8')
        raise
    finally:
        try:
            if core is not None:core.close()
        finally:
            state['fin_unix']=time.time()
            state['fichiers_modeles_inchanges']=all(file_sha256(d['path'])==d['sha256'] and
                file_sha256(d['meta_path'])==d['meta_sha256'] for d in descriptors)
            state['compteurs_inchanges']=all(int(a.num_timesteps)==int(d['meta']['total_timesteps'])
                                            for a,d in zip(agents,descriptors))
            summary=exporter_evaluation(directory,rows,state)
            print('Rapports conserves :',directory)
            display(summary)
    if not state['fichiers_modeles_inchanges'] or not state['compteurs_inchanges']:
        raise RuntimeError('Integrite des modeles modifiee pendant evaluation : voir etat_session.json.')
    return directory


In [14]:
import math

SUP_DEADLINE=float('inf')

class BudgetAtteint(RuntimeError):
    pass


def verifier_plan():
    if type(ACTIVER_SUPERVISEUR) is not bool or type(ACTIVER_SAUVEGARDE_CHECKPOINT) is not bool:
        raise ValueError('Interrupteurs True/False requis.')
    if not re.fullmatch(r'[A-Za-z0-9_-]+',CAMPAGNE):raise ValueError('Nom de campagne invalide.')
    positive=(DUREE_MAX_HEURES,ACTIONS_PAR_CYCLE,EPISODES_EVALUATION,MAX_CYCLES_PAR_LANCEMENT,
              MAX_CYCLES_PAR_LECON,MAX_CYCLES_SANS_PROGRES,EVALUATIONS_CONSECUTIVES_REQUISES,
              TOURS_COMPLETS_MINIMUM_PAR_EVALUATION)
    if not all(np.isfinite(v) and v>0 for v in positive):raise ValueError('Budgets et nombres positifs requis.')
    for count in (ACTIONS_PAR_CYCLE,EPISODES_EVALUATION,MAX_CYCLES_PAR_LANCEMENT,
                  MAX_CYCLES_PAR_LECON,MAX_CYCLES_SANS_PROGRES,EVALUATIONS_CONSECUTIVES_REQUISES,
                  TOURS_COMPLETS_MINIMUM_PAR_EVALUATION,MAX_REESSAIS_TECHNIQUES):
        if type(count) is not int:raise ValueError('Les nombres de cycles/episodes/actions doivent etre entiers.')
    if not 0<REUSSITE_MIN_PCT<=100 or not 0<=SORTIES_MAX_PCT<=100:
        raise ValueError('Seuils de pourcentage invalides.')
    if TOURS_COMPLETS_MINIMUM_PAR_EVALUATION>EPISODES_EVALUATION:
        raise ValueError('Le nombre de tours requis depasse le nombre d episodes evalues.')
    if not 0<GAIN_TEMPS_MIN_PCT<100:raise ValueError('Gain temps entre 0 et 100 requis.')
    if not np.isfinite(INTERVALLE_ACTION_S) or INTERVALLE_ACTION_S<=0 or not np.isfinite(PENALITE_SORTIE_PISTE) or PENALITE_SORTIE_PISTE<=0:
        raise ValueError('Intervalle et penalite positifs requis.')
    if MAX_REESSAIS_TECHNIQUES<0 or not np.isfinite(ATTENTE_REESSAI_S) or ATTENTE_REESSAI_S<0:raise ValueError('Reessais invalides.')
    if not LECONS or LECONS[-1]['stage']!=4 or LECONS[-1]['goal_m'] is not None:
        raise ValueError('La derniere lecon doit etre le tour complet.')
    previous=None;ids=set()
    for lesson in LECONS:
        if not re.fullmatch(r'[A-Za-z0-9_-]+',lesson['id']) or lesson['id'] in ids:
            raise ValueError('Identifiant de lecon invalide ou duplique.')
        ids.add(lesson['id'])
        stage=lesson['stage'];goal=lesson['goal_m']
        if type(stage) is not int or stage not in (1,2,3,4):raise ValueError('Stage invalide.')
        if previous is not None and stage not in (previous,previous+1):raise ValueError('Saut de stage refuse.')
        if stage==4 and goal is not None or stage!=4 and (goal is None or not np.isfinite(goal) or goal<=0):
            raise ValueError('Objectif de lecon invalide.')
        if type(lesson['n_steps']) is not int or lesson['n_steps']<64 or lesson['n_steps']%64 or not np.isfinite(lesson['episode_seconds']) or lesson['episode_seconds']<=0:
            raise ValueError('Lot multiple de 64 et duree positive requis.')
        if lesson['n_steps']*INTERVALLE_ACTION_S<=lesson['episode_seconds']:
            raise ValueError('Le lot doit permettre au moins un episode complet : '+lesson['id'])
        previous=stage
    signature={'source':MODELE_SOURCE,'track':TRACK_ATTENDU,'car':VOITURE_ATTENDUE,
        'direction':DIRECTION_PROGRESSION,'intervalle':INTERVALLE_ACTION_S,
        'penalite':PENALITE_SORTIE_PISTE,'lecons':LECONS,'episodes':EPISODES_EVALUATION,
        'reussite':REUSSITE_MIN_PCT,'sorties':SORTIES_MAX_PCT,
        'consecutives':EVALUATIONS_CONSECUTIVES_REQUISES,
        'tours_min':TOURS_COMPLETS_MINIMUM_PAR_EVALUATION,'gain_temps':GAIN_TEMPS_MIN_PCT}
    return hashlib.sha256(json.dumps(signature,sort_keys=True).encode()).hexdigest()


def configurer_lecon(lesson,model):
    globals().update(ETAPE_APPRENTISSAGE=lesson['stage'],OBJECTIF_DISTANCE_M=lesson['goal_m'],
        MODELE_NOM=model,MODELE_SOURCE_TRANSFERT='',REPRENDRE_MODELE_EXISTANT=True,
        EXIGER_MODELE_EXISTANT=True,DUREE_EPISODE_S=lesson['episode_seconds'],
        PAS_PAR_LOT=lesson['n_steps'],TOTAL_TIMESTEPS=int(math.ceil(max(ACTIONS_PAR_CYCLE,
            lesson['n_steps'])/lesson['n_steps'])*lesson['n_steps']),
        MODELES_A_EVALUER=[model],N_EPISODES=EPISODES_EVALUATION,
        MAX_ACTIONS_PAR_EPISODE=int(math.ceil(lesson['episode_seconds']/INTERVALLE_ACTION_S))+100,
        ACTIVER_EVALUATION=True)


def ecrire_etat(directory,state):
    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)
    temporary=directory/('etat_'+uuid.uuid4().hex+'.tmp')
    temporary.write_text(json.dumps(state,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf8')
    os.replace(str(temporary),str(directory/'etat.json'))


def evenement(directory,state,type_evenement,**detail):
    item={'time':time.time(),'type':type_evenement,**detail}
    with (Path(directory)/'evenements.jsonl').open('a',encoding='utf8') as stream:
        stream.write(json.dumps(item,ensure_ascii=False,allow_nan=False)+'\n')
    state['dernier_evenement']=item
    ecrire_etat(directory,state)
    print(type_evenement,detail)


def notifier(directory,message):
    (Path(directory)/'NOTIFICATION.txt').write_text(message+'\n',encoding='utf8')
    print('NOTIFICATION LOCALE :',message)
    if SIGNAL_SONORE:
        try:
            import winsound
            winsound.MessageBeep()
        except (ImportError,RuntimeError):pass


def evaluation_valide(metrics):
    return (metrics.get('episodes')==EPISODES_EVALUATION
        and metrics.get('episodes_valides')==EPISODES_EVALUATION
        and metrics.get('erreurs_techniques')==0)


def objectif_valide(metrics,lesson):
    if not evaluation_valide(metrics):return False
    passed=(metrics['reussite_pct']>=REUSSITE_MIN_PCT
        and metrics['sortie_pct']<=SORTIES_MAX_PCT
        and metrics['collision_detectee_pct']==0)
    if lesson['stage']==4:
        passed=passed and metrics.get('tours_complets',0)>=TOURS_COMPLETS_MINIMUM_PAR_EVALUATION and metrics.get('tours_invalides',0)==0
    return bool(passed)


def candidat_meilleur(candidate,best):
    if not evaluation_valide(candidate):return False
    if best is None:return True
    s,b=candidate['reussite_pct'],best['reussite_pct']
    def risk(m):return m['sortie_pct']+m['collision_detectee_pct']+100*m.get('tours_invalides',0)/m['episodes_valides']
    r,br=risk(candidate),risk(best)
    if s<b or r>br:return False
    if s>b or r<br:return True
    # Avoid promoting changes that are just a few milliseconds of timing noise.
    ct,bt=candidate.get('temps_median_reussite_s'),best.get('temps_median_reussite_s')
    if ct is not None and bt is not None and ct<=bt*(1-GAIN_TEMPS_MIN_PCT/100):return True
    cp,bp=candidate.get('progression_moyenne_m'),best.get('progression_moyenne_m')
    return bool(s<REUSSITE_MIN_PCT and cp is not None and bp is not None and cp>bp+max(1.,abs(bp)*.05))


def retenter(operation,directory,state,label):
    for attempt in range(MAX_REESSAIS_TECHNIQUES+1):
        if time.monotonic()>=SUP_DEADLINE:raise BudgetAtteint('Budget horaire atteint.')
        try:return operation()
        except (KeyboardInterrupt,BudgetAtteint):raise
        except Exception as exc:
            evenement(directory,state,'incident_technique',operation=label,tentative=attempt+1,erreur=str(exc))
            if attempt>=MAX_REESSAIS_TECHNIQUES:raise
            end=min(SUP_DEADLINE,time.monotonic()+ATTENTE_REESSAI_S)
            while time.monotonic()<end:time.sleep(min(.5,end-time.monotonic()))


def evaluer_agent_local(model,lesson,directory,state):
    configurer_lecon(lesson,model)
    def premier_tour(report):
        if not state['premier_tour_confirme']:
            state['premier_tour_confirme']=True
            state['rapport_premier_tour']=str(report)
            evenement(directory,state,'premier_tour_complet',rapport=str(report),tours=1)
            notifier(directory,'Premier tour complet verifie ! Rapport : '+str(report))
    globals()['SUR_TOUR_CONFIRME']=premier_tour
    try:
        report=retenter(executer_evaluation,directory,state,'evaluation')
    finally:
        globals().pop('SUR_TOUR_CONFIRME',None)
    session=json.loads((Path(report)/'etat_session.json').read_text(encoding='utf8'))
    if session['statut']!='terminee' or not session['fichiers_modeles_inchanges'] or not session['compteurs_inchanges']:
        raise RuntimeError('Rapport incomplet ou integrite non verifiee : '+str(report))
    table=pd.read_csv(Path(report)/'comparaison.csv')
    if len(table)!=1:raise RuntimeError('Une seule ligne attendue pour le candidat.')
    # Convert pandas scalars/NaNs to plain JSON values.
    metrics=json.loads(table.to_json(orient='records'))[0]
    if not evaluation_valide(metrics):raise RuntimeError('Evaluation technique incomplete.')
    if lesson['stage']==4 and metrics.get('tours_complets',0)>0 and not state['premier_tour_confirme']:
        state['premier_tour_confirme']=True
        state['rapport_premier_tour']=str(report)
        evenement(directory,state,'premier_tour_complet',rapport=str(report),tours=metrics['tours_complets'])
        notifier(directory,'Premier tour complet verifie ! Rapport : '+str(report))
    return metrics,str(report)


def appliquer_resultat(state,metrics,report,model,lesson,directory,baseline=False):
    passed=objectif_valide(metrics,lesson)
    improved=baseline or candidat_meilleur(metrics,state.get('best_metrics'))
    archive=archiver_agent(model,'evaluation_superviseur')
    if improved:
        state['best_model']=str(archive.relative_to(PROJECT/'models'))
        state['best_metrics']=metrics;state['best_report']=report;state['sans_progres']=0
    elif not baseline:state['sans_progres']+=1
    state['evaluations_consecutives']=state['evaluations_consecutives']+1 if passed else 0
    evenement(directory,state,'evaluation_terminee',lecon=lesson['id'],modele=model,
              criteres_atteints=passed,conserve=improved,metriques=metrics,rapport=report)
    return passed


def lancer_superviseur_local():
    global SUP_DEADLINE
    signature=verifier_plan()
    directory=PROJECT/'reports'/'superviseur_local'/CAMPAGNE
    state_path=directory/'etat.json'
    if not ACTIVER_SUPERVISEUR:
        print('Superviseur desactive. Parcours :',[l['id'] for l in LECONS])
        if state_path.is_file():display(json.loads(state_path.read_text(encoding='utf8')))
        print('Activer ACTIVER_SUPERVISEUR pour lancer la boucle locale. Aucun appel API/Codex.')
        return None
    SUP_DEADLINE=time.monotonic()+DUREE_MAX_HEURES*3600
    if state_path.is_file():
        state=json.loads(state_path.read_text(encoding='utf8'))
        if state.get('signature')!=signature:raise RuntimeError('Parcours ou criteres modifies : choisir un nouveau nom CAMPAGNE.')
        if state['statut']=='objectif_atteint':
            print('Le tour complet est deja valide. Rapport :',state.get('best_report'));return directory
        # Refuse a modified archived best/source before issuing game commands.
        lire_modele(state['best_model'] or state['lesson_source'])
    else:
        source,meta=lire_modele(MODELE_SOURCE)
        required={'track':TRACK_ATTENDU,'car':VOITURE_ATTENDUE,'control':CONTROL_SCHEMA,
                  'observation':OBS_SCHEMA,'reset':'csp_snapshot_verified','progress_sign':DIRECTION_PROGRESSION}
        if any(meta.get(k)!=v for k,v in required.items()):raise RuntimeError('Source incompatible avec la campagne.')
        if meta.get('curriculum_stage') not in (LECONS[0]['stage']-1,LECONS[0]['stage']):
            raise RuntimeError('Stage source incompatible avec la premiere lecon.')
        source_archive=archiver_agent(source,'debut_campagne')
        state={'version':1,'signature':signature,'campagne':CAMPAGNE,'statut':'initialisation',
            'lesson_index':0,'lesson_source':str(source_archive.relative_to(PROJECT/'models')),
            'best_model':None,'best_metrics':None,'best_report':None,'pending':None,
            'total_cycles':0,'cycles_lecon':0,'evaluations_consecutives':0,'sans_progres':0,
            'premier_tour_confirme':False,'created_at':time.time()}
        ecrire_etat(directory,state)
    start_cycles=state['total_cycles']
    try:
        state['statut']='en_cours';evenement(directory,state,'reprise_lancement')
        while True:
            if time.monotonic()>=SUP_DEADLINE:raise BudgetAtteint('Budget horaire atteint.')
            lesson=LECONS[state['lesson_index']]
            pending=state.get('pending')
            if pending:
                model=pending['model'];configurer_lecon(lesson,model)
                if pending['phase']=='entrainement':
                    _,m=lire_modele(model)
                    if m['total_timesteps']==pending['compteur_initial']:
                        retenter(entrainer_ppo,directory,state,'entrainement')
                    # A completed/partial saved cycle is always evaluated before more training.
                    pending['phase']='evaluation';ecrire_etat(directory,state)
                metrics,report=evaluer_agent_local(model,lesson,directory,state)
                appliquer_resultat(state,metrics,report,model,lesson,directory,pending['baseline'])
                state['pending']=None;ecrire_etat(directory,state)
            if state['evaluations_consecutives']>=EVALUATIONS_CONSECUTIVES_REQUISES:
                if state['lesson_index']==len(LECONS)-1:
                    state['statut']='objectif_atteint'
                    evenement(directory,state,'tour_complet_valide',modele=state['best_model'],rapport=state['best_report'])
                    notifier(directory,'Objectif final atteint : tours complets verifies. '+str(directory))
                    break
                previous=lesson['id'];state['lesson_index']+=1
                state['lesson_source']=state['best_model'];state['best_model']=None
                state['best_metrics']=None;state['best_report']=None
                state['cycles_lecon']=0;state['sans_progres']=0;state['evaluations_consecutives']=0
                evenement(directory,state,'passage_lecon',depuis=previous,vers=LECONS[state['lesson_index']]['id'])
                continue
            if state['sans_progres']>=MAX_CYCLES_SANS_PROGRES:
                state['statut']='stagnation';evenement(directory,state,'arret_stagnation');break
            if state['cycles_lecon']>=MAX_CYCLES_PAR_LECON:
                state['statut']='limite_lecon';evenement(directory,state,'arret_limite_lecon');break
            if state['total_cycles']-start_cycles>=MAX_CYCLES_PAR_LANCEMENT:
                state['statut']='limite_cycles';evenement(directory,state,'arret_limite_cycles');break
            baseline=state['best_model'] is None
            source=state['lesson_source'] if baseline else state['best_model']
            label='sup_'+CAMPAGNE+'_'+lesson['id']+'_'+('base' if baseline else 'c'+str(state['total_cycles']+1))+'_'+uuid.uuid4().hex[:6]
            # A new working agent: never overwrite the best agent or a previous lesson.
            prefix=creer_agent_depuis(source,label,lesson['stage'],lesson['goal_m'])
            _,meta=lire_modele(prefix)
            state['pending']={'model':label,'baseline':baseline,
                'phase':'evaluation' if baseline else 'entrainement','compteur_initial':meta['total_timesteps']}
            if not baseline:
                state['total_cycles']+=1;state['cycles_lecon']+=1
            evenement(directory,state,'candidat_prepare',modele=label,lecon=lesson['id'],baseline=baseline)
    except BudgetAtteint as exc:
        state['statut']='budget_atteint';evenement(directory,state,'arret_budget',detail=str(exc))
    except KeyboardInterrupt:
        state['statut']='interrompu';evenement(directory,state,'interruption_manuelle')
    except Exception as exc:
        state['statut']='erreur_technique'
        evenement(directory,state,'arret_technique',erreur=str(exc))
        (directory/'erreur.txt').write_text(traceback.format_exc(),encoding='utf8')
    finally:
        state['fin_lancement']=time.time();ecrire_etat(directory,state)
        if state['statut']!='objectif_atteint':
            notifier(directory,'Superviseur arrete : '+state['statut']+'. Etat et versions conserves : '+str(directory))
        display(state)
    return directory


## Lancer ou reprendre la campagne

In [15]:
RAPPORT_SUPERVISEUR = lancer_superviseur_local()


reprise_lancement {}
Agent cree : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Code\models\sup_vers_un_tour_01_30m_base_68b2dd.zip | stage 2 | compteur herite 135424
Source conservee : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Code\models\versions\ppo_vision_checkpoint_v3_stage2\20261117_111045_b5ca829b\agent
Version initiale conservee : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Code\models\versions\sup_vers_un_tour_01_30m_base_68b2dd\20261117_114147_3ae11fad\agent
candidat_prepare {'modele': 'sup_vers_un_tour_01_30m_base_68b2dd', 'lecon': '30m', 'baseline': True}
Modeles verifies :
 - sup_vers_un_tour_01_30m_base_68b2dd | compteur apprentissage : 135424
Application : ('PONG', 'OUARTI CSP app active - V3.2')
Capacites : {'full_snapshot': True, 'position_fallback': False, 'offline': True}
Sauvegarde du checkpoint CSP...
Checkpoint enregistre : {'track': 'ks_barcelona-layout_gp', 'car': 'abarth500', 'x': -538.3145751953125, 'y': 172.0554046

,modele,episodes,episodes_valides,erreurs_techniques,reussite_pct,sortie_pct,collision_detectee_pct,immobilisation_pct,limite_pct,tours_complets,tours_invalides,temps_median_reussite_s,progression_moyenne_m,reward_moyenne,volant_sature_pct,variation_volant_moyenne,vitesse_max_kmh
0,sup_vers_un_tour_01_30m_base_68b2dd,10,10,0,0.0,100.0,0.0,0.0,0.0,0,0,None,17.750315,-82.866085,0.0,0.00038,42.772776


evaluation_terminee {'lecon': '30m', 'modele': 'sup_vers_un_tour_01_30m_base_68b2dd', 'criteres_atteints': False, 'conserve': True, 'metriques': {'modele': 'sup_vers_un_tour_01_30m_base_68b2dd', 'episodes': 10, 'episodes_valides': 10, 'erreurs_techniques': 0, 'reussite_pct': 0.0, 'sortie_pct': 100.0, 'collision_detectee_pct': 0.0, 'immobilisation_pct': 0.0, 'limite_pct': 0.0, 'tours_complets': 0, 'tours_invalides': 0, 'temps_median_reussite_s': None, 'progression_moyenne_m': 17.7503151706, 'reward_moyenne': -82.8660850168, 'volant_sature_pct': 0.0, 'variation_volant_moyenne': 0.0003798756, 'vitesse_max_kmh': 42.7727760315}, 'rapport': 'C:\\Users\\elkat\\Documents\\Github\\IA_projet_Ouarti_ds_github\\Code\\reports\\evaluation_v3\\20261117_114152_b2e89862'}
Agent cree : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Code\models\sup_vers_un_tour_01_30m_c1_fa5612.zip | stage 2 | compteur herite 135424
Source conservee : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github

,modele,episodes,episodes_valides,erreurs_techniques,reussite_pct,sortie_pct,collision_detectee_pct,immobilisation_pct,limite_pct,tours_complets,tours_invalides,temps_median_reussite_s,progression_moyenne_m,reward_moyenne,volant_sature_pct,variation_volant_moyenne,vitesse_max_kmh
0,sup_vers_un_tour_01_30m_c1_fa5612,10,10,0,100.0,0.0,0.0,0.0,0.0,0,0,4.5785,31.306265,60.676651,0.0,0.001481,59.323686


evaluation_terminee {'lecon': '30m', 'modele': 'sup_vers_un_tour_01_30m_c1_fa5612', 'criteres_atteints': True, 'conserve': True, 'metriques': {'modele': 'sup_vers_un_tour_01_30m_c1_fa5612', 'episodes': 10, 'episodes_valides': 10, 'erreurs_techniques': 0, 'reussite_pct': 100.0, 'sortie_pct': 0.0, 'collision_detectee_pct': 0.0, 'immobilisation_pct': 0.0, 'limite_pct': 0.0, 'tours_complets': 0, 'tours_invalides': 0, 'temps_median_reussite_s': 4.5785, 'progression_moyenne_m': 31.3062645189, 'reward_moyenne': 60.6766513278, 'volant_sature_pct': 0.0, 'variation_volant_moyenne': 0.0014808348, 'vitesse_max_kmh': 59.3236862183}, 'rapport': 'C:\\Users\\elkat\\Documents\\Github\\IA_projet_Ouarti_ds_github\\Code\\reports\\evaluation_v3\\20261117_120216_882be933'}
Agent cree : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Code\models\sup_vers_un_tour_01_30m_c2_f45a17.zip | stage 2 | compteur herite 139520
Source conservee : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Cod

,modele,episodes,episodes_valides,erreurs_techniques,reussite_pct,sortie_pct,collision_detectee_pct,immobilisation_pct,limite_pct,tours_complets,tours_invalides,temps_median_reussite_s,progression_moyenne_m,reward_moyenne,volant_sature_pct,variation_volant_moyenne,vitesse_max_kmh
0,sup_vers_un_tour_01_30m_c2_f45a17,10,10,0,100.0,0.0,0.0,0.0,0.0,0,0,4.6095,31.460908,60.793786,0.0,0.002721,59.635272


... (8 outputs left out)

evaluation_terminee {'lecon': '100m', 'modele': 'sup_vers_un_tour_01_100m_c5_4c4163', 'criteres_atteints': False, 'conserve': False, 'metriques': {'modele': 'sup_vers_un_tour_01_100m_c5_4c4163', 'episodes': 10, 'episodes_valides': 10, 'erreurs_techniques': 0, 'reussite_pct': 0.0, 'sortie_pct': 100.0, 'collision_detectee_pct': 0.0, 'immobilisation_pct': 0.0, 'limite_pct': 0.0, 'tours_complets': 0, 'tours_invalides': 0, 'temps_median_reussite_s': None, 'progression_moyenne_m': 92.7812976026, 'reward_moyenne': -8.4704157126, 'volant_sature_pct': 0.0, 'variation_volant_moyenne': 0.0063971255, 'vitesse_max_kmh': 94.7341323853}, 'rapport': 'C:\\Users\\elkat\\Documents\\Github\\IA_projet_Ouarti_ds_github\\Code\\reports\\evaluation_v3\\20261117_125227_b2aae569'}
Agent cree : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Code\models\sup_vers_un_tour_01_100m_c6_580592.zip | stage 3 | compteur herite 139520
Source conservee : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github

,modele,episodes,episodes_valides,erreurs_techniques,reussite_pct,sortie_pct,collision_detectee_pct,immobilisation_pct,limite_pct,tours_complets,tours_invalides,temps_median_reussite_s,progression_moyenne_m,reward_moyenne,volant_sature_pct,variation_volant_moyenne,vitesse_max_kmh
0,sup_vers_un_tour_01_100m_c6_580592,10,10,0,0.0,100.0,0.0,0.0,0.0,0,0,None,33.766476,-67.043987,0.0,0.002942,53.427001


evaluation_terminee {'lecon': '100m', 'modele': 'sup_vers_un_tour_01_100m_c6_580592', 'criteres_atteints': False, 'conserve': False, 'metriques': {'modele': 'sup_vers_un_tour_01_100m_c6_580592', 'episodes': 10, 'episodes_valides': 10, 'erreurs_techniques': 0, 'reussite_pct': 0.0, 'sortie_pct': 100.0, 'collision_detectee_pct': 0.0, 'immobilisation_pct': 0.0, 'limite_pct': 0.0, 'tours_complets': 0, 'tours_invalides': 0, 'temps_median_reussite_s': None, 'progression_moyenne_m': 33.7664757324, 'reward_moyenne': -67.0439870399, 'volant_sature_pct': 0.0, 'variation_volant_moyenne': 0.0029418074, 'vitesse_max_kmh': 53.4270011902}, 'rapport': 'C:\\Users\\elkat\\Documents\\Github\\IA_projet_Ouarti_ds_github\\Code\\reports\\evaluation_v3\\20261117_130452_357280c8'}
arret_stagnation {}
NOTIFICATION LOCALE : Superviseur arrete : stagnation. Etat et versions conserves : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Code\reports\superviseur_local\vers_un_tour_01


{'version': 1,
 'signature': '7339eb39ab20891d8e404348c82b71b35e82db310c36ef54cdf5f6ff76e6aa0c',
 'campagne': 'vers_un_tour_01',
 'statut': 'stagnation',
 'lesson_index': 1,
 'lesson_source': 'versions\\sup_vers_un_tour_01_30m_c1_fa5612\\20261117_120215_d0e6718d\\agent',
 'best_model': 'versions\\sup_vers_un_tour_01_100m_base_a6b1e7\\20261117_121531_af48aece\\agent',
 'best_metrics': {'modele': 'sup_vers_un_tour_01_100m_base_a6b1e7',
  'episodes': 10,
  'episodes_valides': 10,
  'erreurs_techniques': 0,
  'reussite_pct': 0.0,
  'sortie_pct': 100.0,
  'collision_detectee_pct': 0.0,
  'immobilisation_pct': 0.0,
  'limite_pct': 0.0,
  'tours_complets': 0,
  'tours_invalides': 0,
  'temps_median_reussite_s': None,
  'progression_moyenne_m': 94.6412137473,
  'reward_moyenne': -6.5220970476,
  'volant_sature_pct': 0.0,
  'variation_volant_moyenne': 0.0027694931,
  'vitesse_max_kmh': 95.3028671265},
 'best_report': 'C:\\Users\\elkat\\Documents\\Github\\IA_projet_Ouarti_ds_github\\Code\\report